# Anatomical subtissue identity in the GTEx CLAMP latent space

This analysis asks whether the complete per-sample CLAMP latent-variable (LV) space derived from bulk GTEx RNA-seq preserves anatomical identity within broad tissues. The final analysis uses all 578 CLAMP latent variables; feature-attribution representations are not included in the reported results.

All results use the same fixed five-fold donor-grouped splits. The production evaluator performs the downstream logistic-regression fits, donor-profile permutation tests, and donor bootstraps; this notebook validates and visualizes the full-latent-space results.

💡 **Environment:** `clamp-analyses`

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyprojroot.here import here

OUTPUT_DIR = here("output/03_model_biology/01_gtex/04_subtissues")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

try:
    SNAKEMAKE = snakemake
except NameError:
    SNAKEMAKE = None

mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 10,
    "axes.labelsize": 10.5,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

FULLLV_COLOR = "#DD8452"
FULLLV_MARKER = "o"
FULLLV_LABEL = "Full CLAMP latent space (578 LVs)"

OUTPUT_STEMS = {
    "main": "anatomical_subtissue_panel",
    "supp1": "supp1_all_smtsd_performance",
    "supp2": "supp2_metrics",
    "supp3": "supp3_confusion_matrices",
    "supp4": "supp4_permutation_nulls",
    "supp5": "supp5_subtissue_counts",
}

def save_figure(fig, key):
    for extension in ("png", "pdf", "svg"):
        if SNAKEMAKE is not None:
            path = Path(SNAKEMAKE.output[f"{key}_{extension}"])
        else:
            path = FIGURE_DIR / f"{OUTPUT_STEMS[key]}.{extension}"
        path.parent.mkdir(parents=True, exist_ok=True)
        kwargs = {"dpi": 300} if extension == "png" else {}
        fig.savefig(path, bbox_inches="tight", facecolor="white", **kwargs)

def clean_axes(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_axisbelow(True)

def p_text(row):
    exceedances = int(row["FullLV_Permutation_Exceedances"])
    repeats = int(row["FullLV_N_Permutations"])
    p_value = float(row["FullLV_Permutation_P_Value"])
    if exceedances == 0:
        return f"P ≤ {1 / (repeats + 1):.3f}"
    return f"P = {p_value:.3g}"

def full_lv_performance_plot(data, ax, annotate_raw=True):
    plot_data = data.sort_values("FullLV_Adjusted_Balanced_Accuracy", ascending=False).reset_index(drop=True)
    y = np.arange(len(plot_data))
    values = plot_data["FullLV_Adjusted_Balanced_Accuracy"].to_numpy()
    lower = plot_data["FullLV_Adjusted_BA_CI_Lower"].to_numpy()
    upper = plot_data["FullLV_Adjusted_BA_CI_Upper"].to_numpy()
    ax.errorbar(
        values, y, xerr=np.vstack([values - lower, upper - values]),
        fmt=FULLLV_MARKER, markersize=6.5, color=FULLLV_COLOR,
        markeredgecolor="white", markeredgewidth=0.7, ecolor=FULLLV_COLOR,
        elinewidth=1.2, capsize=2.3, zorder=3,
    )
    if annotate_raw:
        for x_value, y_value, raw_ba in zip(
            values, y, plot_data["FullLV_Balanced_Accuracy"],
        ):
            ax.text(
                min(x_value + 0.018, 1.005), y_value, f"BA {raw_ba:.2f}",
                va="center", ha="left", fontsize=7.0, color=FULLLV_COLOR,
            )
    ax.axvline(0, color="#555555", linestyle=(0, (3, 2)), linewidth=0.9)
    ax.set_xlim(-0.02, 1.10 if annotate_raw else 1.02)
    ax.set_xticks(np.arange(0, 1.01, 0.2))
    ax.set_yticks(y)
    ax.set_yticklabels(plot_data["Tissue_Display"])
    ax.invert_yaxis()
    ax.set_xlabel("Adjusted balanced accuracy")
    ax.xaxis.grid(True, color="#E5E5E5", linewidth=0.7)
    clean_axes(ax)
    return plot_data

def confusion_matrix(confusion, analysis, tissue, representation):
    subset = confusion[
        (confusion["Analysis"] == analysis)
        & (confusion["Tissue"] == tissue)
        & (confusion["Representation"] == representation)
    ].copy()
    order = subset.drop_duplicates("True_SMTSD")[["True_SMTSD", "True_Display"]]
    labels = order["True_SMTSD"].tolist()
    display = order.set_index("True_SMTSD")["True_Display"].to_dict()
    matrix = subset.pivot(index="True_SMTSD", columns="Predicted_SMTSD", values="Row_Proportion")
    matrix = matrix.reindex(index=labels, columns=labels).fillna(0.0)
    matrix.index = [display[label] for label in labels]
    matrix.columns = [display[label] for label in labels]
    return matrix

def draw_confusion(matrix, ax, title, cbar=False, cbar_ax=None):
    n_classes = len(matrix)
    annotations = np.full(matrix.shape, "", dtype=object)
    for row in range(n_classes):
        for column in range(n_classes):
            value = matrix.iloc[row, column]
            if n_classes <= 3 or row == column or value >= 0.10:
                annotations[row, column] = f"{value:.0%}"
    sns.heatmap(
        matrix, annot=annotations, fmt="", cmap="YlGn", vmin=0, vmax=1,
        linewidths=0.35, linecolor="white", square=True, ax=ax,
        cbar=cbar, cbar_ax=cbar_ax, annot_kws={"fontsize": 6.4},
    )
    ax.set_title(title, fontsize=10.5)
    ax.set_xlabel("Predicted region")
    ax.set_ylabel("True region")
    ax.tick_params(axis="x", rotation=52, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)


## Production results and validation

In [ ]:
anatomical = pd.read_csv(OUTPUT_DIR / "anatomical_subtissue_results.tsv", sep="\t")
all_smtsd = pd.read_csv(OUTPUT_DIR / "all_smtsd_results.tsv", sep="\t")
supplementary_table = pd.read_csv(OUTPUT_DIR / "subtissue_supplementary_table.tsv", sep="\t")
confusion = pd.read_csv(OUTPUT_DIR / "subtissue_confusion_matrices.tsv", sep="\t").query("Representation == 'FullLV'").copy()
permutation_nulls = pd.read_csv(OUTPUT_DIR / "subtissue_permutation_nulls.tsv", sep="\t").query("Representation == 'FullLV'").copy()
counts = pd.read_csv(OUTPUT_DIR / "subtissue_counts.tsv", sep="\t")
exclusions = pd.read_csv(OUTPUT_DIR / "subtissue_exclusions.tsv", sep="\t")
fold_audit = pd.read_csv(OUTPUT_DIR / "fold_smtsd_audit.tsv", sep="\t")

expected_tissues = {"Adipose Tissue", "Blood Vessel", "Brain", "Colon", "Esophagus", "Heart", "Skin"}
assert set(anatomical["Tissue"]) == expected_tissues
assert len(anatomical) == 7 and len(all_smtsd) == 8
assert anatomical.loc[anatomical["Tissue"] == "Brain", "N_Subtissue_Classes"].item() == 13
assert not counts.query("Included_Anatomical")["SMTSD"].eq("Cells - Cultured fibroblasts").any()
assert (fold_audit.query("Included_All_SMTSD")["N_Samples"] > 0).all()
assert np.allclose(
    anatomical["FullLV_Adjusted_Balanced_Accuracy"],
    (anatomical["FullLV_Balanced_Accuracy"] - 1 / anatomical["N_Subtissue_Classes"])
    / (1 - 1 / anatomical["N_Subtissue_Classes"]),
)

display(anatomical[[
    "Tissue_Display", "N_Samples", "N_Donors",
    "FullLV_Balanced_Accuracy", "FullLV_Macro_F1",
]])


## Main panel

Panel A reports full-latent-space performance on chance-adjusted balanced accuracy, allowing binary, three-class, and 13-class tasks to share a meaningful scale. Panel B shows every Brain region; performance is not driven by a small subset of easy classes. Raw balanced accuracy is printed beside each point.

In [ ]:
fig = plt.figure(figsize=(16.2, 7.6), constrained_layout=False)
grid = fig.add_gridspec(1, 2, width_ratios=[0.88, 1.62], wspace=0.34)
ax_performance = fig.add_subplot(grid[0, 0])
ax_brain = fig.add_subplot(grid[0, 1])

ordered = full_lv_performance_plot(anatomical, ax_performance, annotate_raw=True)
ax_performance.set_title(
    "Anatomical subtissue identity", loc="left", fontweight="bold", y=1.075,
)
above_chance = int((anatomical["FullLV_Adjusted_Balanced_Accuracy"] > 0).sum())
median_full = anatomical["FullLV_Balanced_Accuracy"].median()
all_floor = anatomical["FullLV_Permutation_Exceedances"].eq(0).all()
max_p = anatomical["FullLV_Permutation_P_Value"].max()
p_summary = (
    f"All empirical P ≤ {max_p:.3f}"
    if all_floor else f"Maximum empirical P = {max_p:.3g}"
)
ax_performance.text(
    0.01, 1.005,
    f"{above_chance}/7 tissues above chance  |  "
    f"Median Full-LV BA = {median_full:.3f}  |  {p_summary}",
    transform=ax_performance.transAxes, fontsize=7.6, va="bottom",
)

brain_matrix = confusion_matrix(confusion, "anatomical", "Brain", "FullLV")
brain_row = anatomical.set_index("Tissue").loc["Brain"]
draw_confusion(
    brain_matrix, ax_brain,
    f"Brain: full CLAMP LV space (13 regions)\n"
    f"balanced accuracy = {brain_row['FullLV_Balanced_Accuracy']:.3f}; "
    f"macro-F1 = {brain_row['FullLV_Macro_F1']:.3f}",
    cbar=True,
)
ax_brain.collections[0].colorbar.set_label("Fraction of true region", fontsize=9)

fig.text(0.018, 0.965, "A", fontsize=16, fontweight="bold")
fig.text(0.407, 0.965, "B", fontsize=16, fontweight="bold")
fig.subplots_adjust(left=0.18, right=0.98, bottom=0.18, top=0.89)
save_figure(fig, "main")
plt.show()


## Supplementary figures

The complete detailed-label analysis is described as recovery of GTEx `SMTSD` labels rather than strictly anatomical subtissues. Blood and cultured fibroblasts are retained only here.

In [ ]:
# Supplementary figure 1: every eligible detailed-label comparison.
fig, ax = plt.subplots(figsize=(9.2, 6.2))
full_lv_performance_plot(all_smtsd, ax, annotate_raw=True)
ax.set_title("Detailed GTEx tissue-label recovery in the full CLAMP latent space", loc="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, "supp1")
plt.show()

# Supplementary figure 2: raw balanced accuracy and macro-F1.
metric_order = all_smtsd.sort_values("FullLV_Balanced_Accuracy", ascending=False).reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.7), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, metric, title in zip(axes, ("Balanced_Accuracy", "Macro_F1"), ("Balanced accuracy", "Macro-F1")):
    y = np.arange(len(metric_order))
    ax.scatter(
        metric_order[f"FullLV_{metric}"], y, s=42,
        marker=FULLLV_MARKER, color=FULLLV_COLOR, edgecolor="white",
        linewidth=0.7, label=FULLLV_LABEL, zorder=3,
    )
    if metric == "Balanced_Accuracy":
        ax.scatter(
            1 / metric_order["N_Subtissue_Classes"], y, marker="|", s=95,
            color="#555555", linewidth=1.2, label="Chance (1/K)", zorder=4,
        )
    ax.set_xlim(0, 1.02)
    ax.set_xlabel(title)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_yticks(y)
    ax.set_yticklabels(metric_order["Tissue_Display"])
    ax.invert_yaxis()
    ax.xaxis.grid(True, color="#E5E5E5", linewidth=0.7)
    clean_axes(ax)
axes[0].legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
save_figure(fig, "supp2")
plt.show()

# Supplementary figure 3: full-latent-space anatomical confusion matrices.
confusion_order = ["Adipose Tissue", "Blood Vessel", "Colon", "Esophagus", "Heart", "Skin"]
fig = plt.figure(figsize=(13, 20))
grid = fig.add_gridspec(4, 2, height_ratios=[2.25, 1, 1, 1], hspace=0.95, wspace=0.48)
brain_result = anatomical.set_index("Tissue").loc["Brain"]
brain_ax = fig.add_subplot(grid[0, :])
draw_confusion(
    confusion_matrix(confusion, "anatomical", "Brain", "FullLV"), brain_ax,
    f"{brain_result['Tissue_Display']} — full CLAMP latent space\n"
    f"BA = {brain_result['FullLV_Balanced_Accuracy']:.3f}; "
    f"macro-F1 = {brain_result['FullLV_Macro_F1']:.3f}",
    cbar=False,
)
for panel_index, tissue in enumerate(confusion_order):
    matrix = confusion_matrix(confusion, "anatomical", tissue, "FullLV")
    result = anatomical.set_index("Tissue").loc[tissue]
    ax = fig.add_subplot(grid[1 + panel_index // 2, panel_index % 2])
    draw_confusion(
        matrix, ax,
        f"{result['Tissue_Display']} — full CLAMP latent space\n"
        f"BA = {result['FullLV_Balanced_Accuracy']:.3f}; "
        f"macro-F1 = {result['FullLV_Macro_F1']:.3f}",
        cbar=False,
    )
fig.subplots_adjust(left=0.10, right=0.90, bottom=0.05, top=0.98)
colorbar_ax = fig.add_axes([0.93, 0.37, 0.012, 0.26])
normalizer = mpl.colors.Normalize(vmin=0, vmax=1)
colorbar = fig.colorbar(mpl.cm.ScalarMappable(norm=normalizer, cmap="YlGn"), cax=colorbar_ax)
colorbar.set_label("Fraction of true subtissue")
save_figure(fig, "supp3")
plt.show()

# Supplementary figure 4: donor-aware permutation nulls.
null_order = anatomical.sort_values("Tissue_Order")["Tissue"].tolist()
fig, axes = plt.subplots(len(null_order), 1, figsize=(8.5, 20), squeeze=False)
for row_index, tissue in enumerate(null_order):
    result = anatomical.set_index("Tissue").loc[tissue]
    ax = axes[row_index, 0]
    values = permutation_nulls.query(
        "Analysis == 'anatomical' and Tissue == @tissue"
    )["Balanced_Accuracy"]
    observed = result["FullLV_Balanced_Accuracy"]
    sns.histplot(values, bins=30, color=FULLLV_COLOR, alpha=0.72, ax=ax)
    ax.axvline(observed, color="black", linewidth=1.3, linestyle=(0, (4, 2)))
    ax.set_title(
        f"{result['Tissue_Display']} — full CLAMP latent space\n"
        f"observed BA = {observed:.3f}; {p_text(result)}",
        fontsize=9.5,
    )
    ax.set_xlabel("Permuted balanced accuracy")
    ax.set_ylabel("Permutations")
    clean_axes(ax)
fig.tight_layout(h_pad=2.0)
save_figure(fig, "supp4")
plt.show()

# Supplementary figure 5: sample and donor counts, including exclusions.
count_tissue_order = ["Adipose Tissue", "Blood Vessel", "Brain", "Colon", "Esophagus", "Heart", "Skin", "Blood", "Kidney"]
count_data = counts.assign(
    Tissue_Order=counts["Tissue"].map({tissue: index for index, tissue in enumerate(count_tissue_order)}),
    Plot_Label=counts["Tissue"] + " — " + counts["Class_Display"],
).sort_values(["Tissue_Order", "Tissue", "SMTSD"]).reset_index(drop=True)
y = np.arange(len(count_data))
fig, ax = plt.subplots(figsize=(11.5, 13))
for index, row in count_data.iterrows():
    ax.plot([row["N_Donors"], row["N_Samples"]], [index, index], color="#B7B7B7", linewidth=0.8)
ax.scatter(count_data["N_Samples"], y - 0.10, s=34, color="#007A33", label="Samples", zorder=3)
ax.scatter(count_data["N_Donors"], y + 0.10, s=31, marker="D", color="#6A51A3", label="Donors", zorder=3)
ax.axvline(20, color="#555555", linestyle=(0, (3, 2)), linewidth=0.9, label="Minimum samples")
ax.set_yticks(y)
ax.set_yticklabels(count_data["Plot_Label"], fontsize=7.3)
ax.invert_yaxis()
ax.set_xlabel("Count")
ax.set_title("GTEx detailed-label sample and donor counts", loc="left", fontweight="bold")
ax.xaxis.grid(True, color="#E5E5E5", linewidth=0.7)
clean_axes(ax)
ax.legend(frameon=False, ncol=3, loc="lower right")
status_colors = {
    "anatomical_primary": "#222222",
    "supplementary_only": "#4C72B0",
    "excluded_below_min_samples": "#B2182B",
    "excluded_no_within_tissue_task": "#B2182B",
}
for tick, status in zip(ax.get_yticklabels(), count_data["Analysis_Status"]):
    tick.set_color(status_colors[status])
fig.tight_layout()
save_figure(fig, "supp5")
plt.show()


## Results summary

The complete 578-dimensional CLAMP latent space strongly preserves anatomical subtissue identity. All seven anatomical tasks perform above chance with empirical $P \leq 0.001$, and all 13 Brain regions remain well resolved.

In [ ]:
summary_columns = [
    "Tissue_Display", "N_Subtissue_Classes", "N_Samples", "N_Donors",
    "FullLV_Balanced_Accuracy", "FullLV_Adjusted_Balanced_Accuracy",
    "FullLV_Macro_F1", "FullLV_BA_CI_Lower", "FullLV_BA_CI_Upper",
    "FullLV_Permutation_P_Value",
]
display(anatomical[summary_columns])
display(exclusions)
